# NEXUS Trading Agent — LoRA Fine-tuning

**Setup:** Runtime → Change runtime type → T4 GPU (free)

**Steps:**
1. Run all cells in order
2. Upload your `training_data.jsonl` when prompted
3. Wait ~30-60 min for training
4. Download the GGUF file and put it in LM Studio

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth
!pip install --upgrade triton

In [ ]:
# Cell 2: Upload training data
from google.colab import files
uploaded = files.upload()  # Upload your training_data.jsonl

import os
data_file = list(uploaded.keys())[0]
print(f'Uploaded: {data_file}')

# Count examples
with open(data_file) as f:
    num_examples = sum(1 for _ in f)
print(f'Training examples: {num_examples}')

In [ ]:
# Cell 3: Load model with unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-7B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=128,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# Cell 4: Prepare dataset
from datasets import load_dataset

def format_example(example):
    messages = example['messages']
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

dataset = load_dataset('json', data_files=data_file, split='train')
dataset = dataset.map(format_example)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f'Train: {len(train_dataset)} examples')
print(f'Eval:  {len(eval_dataset)} examples')

In [ ]:
# Cell 5: Train!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir='checkpoints',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='steps',
        save_steps=500,
        save_total_limit=2,
        optim='adamw_8bit',
        seed=42,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

stats = trainer.train()
print(f'Training loss: {stats.training_loss:.4f}')

In [ ]:
# Cell 6: Export GGUF for LM Studio
model.save_pretrained_gguf(
    'nexus_trader_gguf',
    tokenizer,
    quantization_method='q4_k_m',
)
print('GGUF export complete!')

In [ ]:
# Cell 7: Download the GGUF file
import glob
from google.colab import files

gguf_files = glob.glob('nexus_trader_gguf/*.gguf')
if gguf_files:
    print(f'Downloading: {gguf_files[0]}')
    files.download(gguf_files[0])
else:
    print('No GGUF file found. Check Cell 6 output.')

# Also save the LoRA adapter (small, ~100MB) as backup
model.save_pretrained('nexus_lora_adapter')
tokenizer.save_pretrained('nexus_lora_adapter')

import shutil
shutil.make_archive('nexus_lora_adapter', 'zip', 'nexus_lora_adapter')
files.download('nexus_lora_adapter.zip')
print('LoRA adapter downloaded as backup!')

## After downloading:

1. **Copy the `.gguf` file** to your LM Studio models folder:
   `C:/Users/Jxint/.lmstudio/models/`

2. **In LM Studio:**
   - Go to My Models
   - The GGUF should appear automatically
   - Select it and test a chat

3. **In NEXUS Settings:**
   - Provider: LM Studio
   - Model: the GGUF filename (e.g. `nexus-trader-Q4_K_M.gguf`)

Your fine-tuned model will now make trading decisions
trained on 52K+ real market outcomes!